# Computational Lab: Density Matrices and Quantum Coherence

**Student(s):** [Name]  
**Class / Period:** [Period]  
**Date:** [Date]

**Instructions:** Complete this notebook as you work through Parts 0–3 of the lab. Write predictions **before running** the relevant simulations. Fill in every marked code cell, insert the four required plots at their designated locations, and write your own answers to the six physics questions. You may use an LLM for coding, but **not to write predictions, verification interpretations, or physics analysis responses**.

**Submission:** Submit this completed `.ipynb` file only. Keep cells in order; use **Run All** before submission. Your notebook must run from top to bottom after dependencies are installed.

**Notebook conventions:** `TODO` indicates student code; `[Write here]` indicates student-written Markdown. The illustrative apparatus figures are embedded in this notebook and are **not** substitutes for Figures 1–4 that you must generate.

## Part 0 — Setup and Analytical Warm-Up

### Choose your coding environment

- **Google Colab:** Open [colab.research.google.com](https://colab.research.google.com), choose **Upload**, and select this `.ipynb` notebook. Run the installation cell once per new runtime.
- **School OnDemand Jupyter:** Upload this notebook to your OnDemand Jupyter file browser, open it, and select the school-supported Python kernel. Check whether QuTiP is installed before installing anything. If installing packages is restricted, ask the instructor/administrator to provide the environment.
- **VS Code:** Install Python and the VS Code **Python** and **Jupyter** extensions. Open this notebook, choose a Python environment/kernel, and install dependencies in that selected environment (for example, in a terminal using `python -m pip install qutip numpy matplotlib`).

**Dependencies:** Python 3, QuTiP, NumPy, Matplotlib. Do not install packages from untrusted sources. If the environment already supplies them, skip installation.

In [ ]:
# Run this cell only if qutip / numpy / matplotlib are not available.
# Colab: uncomment the next line if needed.
# %pip install qutip numpy matplotlib

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

up = qt.basis(2, 0)
down = qt.basis(2, 1)

print("QuTiP version:", qt.__version__)
print("NumPy version:", np.__version__)

---
## Part 1 — Constructing Mixed States
### Activity 1: Phase distributions and density matrices

**Goals:** Average pure-state projectors; investigate coherence; display the individual Bloch-sphere states and their ensemble vector.

For the equatorial state

$$|\psi(\phi)\rangle=\frac1{\sqrt2}(|+z\rangle+e^{i\phi}|-z\rangle),$$

sample $N=1000$ phases uniformly from $[-\Delta,\Delta]$, with $\Delta=0,\pi/2,\pi$. The analytic coherence is

$$|\rho_{12}|=\frac12\left|\frac{\sin\Delta}{\Delta}\right|,$$

using the continuous limit at $\Delta=0$.

### Initial Predictions — Complete before simulation

**Prediction A (density matrix):** As $\Delta$ increases, which density-matrix entries change and which stay constant? Why?

**Prediction A**

> [Write your prediction before running the simulation.]

**Prediction B (Bloch sphere):** Each constituent state lies on the sphere surface. Where will the *ensemble average* vector lie for the three phase spreads?

**Prediction B**

> [Write your prediction and reasoning before running the simulation.]

### Code Stem — State generation and ensemble averaging

In [ ]:
def make_state(phi):
    """Equatorial pure state with relative phase phi (radians)."""
    return (up + np.exp(1j * phi) * down).unit()


def make_ensemble(N, delta, seed=42):
    """Average N pure-state projectors over phases in [-delta, delta]."""
    rng = np.random.default_rng(seed)
    phases = rng.uniform(-delta, delta, N)
    states = [make_state(phi) for phi in phases]
    rho = sum((qt.ket2dm(s) for s in states), start=0) / N
    return rho, states, phases

rho, states, phases = make_ensemble(1000, np.pi / 2)
print(rho)
print("Trace:", rho.tr())
print("Hermitian:", rho.isherm)
print("Eigenvalues:", rho.eigenenergies())

### Figure 1 — Bloch sphere: three phase spreads

Show a representative subset of individual pure states and the **average Bloch vector** for $\Delta=0,\pi/2,\pi$. Label the panels or views.

In [ ]:
def bloch_vector(rho):
    return np.array([
        qt.expect(qt.sigmax(), rho),
        qt.expect(qt.sigmay(), rho),
        qt.expect(qt.sigmaz(), rho)
    ], dtype=float)

print("Average Bloch vector:", bloch_vector(rho))

# TODO: Use qt.Bloch() to plot representative pure states and
# ensemble vectors for the three phase spreads.
# Use a subset of the 1000 states to avoid clutter.

**Figure 1 caption**

> [Insert a figure caption explaining the individual states and average vector. Your plot should appear above this cell.]

### Figure 2 — Off-diagonal coherence versus phase spread

Sweep $\Delta$ from $0$ to $\pi$; plot numerical $|\rho_{12}|$ and overlay the analytical result.

In [ ]:
deltas = np.linspace(0, np.pi, 31)
analytic_coherence = 0.5 * np.abs(np.sinc(deltas / np.pi))
# np.sinc(x) = sin(pi*x)/(pi*x), including the limit at zero.

# TODO: create ensembles at each delta, record abs(rho.full()[0, 1]).
# TODO: generate Figure 2 with numerical points and analytic curve.

**Figure 2 caption**

> [Summarize the trend and any sampling fluctuations.]

### Verification — Part 1

- Confirm that individual pure-state Bloch vectors have magnitude 1 (within numerical precision).
- Record $|\mathbf r|$ for each ensemble and check the original Prediction B.
- Check the diagonal values, trace, Hermiticity, and nonnegative eigenvalues.
- At $\Delta=0,\pi/2,\pi$, compare numerical $|\rho_{12}|$ with theory; give **one nontrivial** numerical/analytic difference.
- Record the **three numerical density matrices**.

In [ ]:
# TODO: print three density matrices and a compact comparison table.
# TODO: calculate one absolute error |simulation - theory|.

**Verification and predictions revisited — Part 1**

> [Report your quantitative checks; indicate which predictions held and what you learned.]

### Physics Analysis — Part 1 (write independently)

**Question 1.** Every constituent state gives $P(+z)=P(-z)=1/2$. Why do ensemble density matrices change as $\Delta$ increases? Which entries change and which remain constant?

**Answer to Question 1**

> Write your response here.

**Question 2.** Why does the off-diagonal element approach zero for phases distributed over the entire circle, even though each constituent state is pure? Explain in terms of averaging complex numbers. Must a mixed state always have zero off-diagonal entries?

**Answer to Question 2**

> Write your response here.

---
## Part 2 — Measuring Coherence
### Activity 2: Purity and the density-matrix Born rule

Reuse the density matrices from Part 1 to calculate

$$\mathcal P=\mathrm{Tr}(\rho^2),\qquad P(+j)=\mathrm{Tr}(\rho P_{+j}),\quad j=x,y,z.$$

For the symmetric equatorial phase ensemble, with $\gamma=\sin\Delta/\Delta$,

$$\mathcal P=\frac12(1+\gamma^2),\quad P(+x)=\frac12(1+\gamma),\quad P(+y)=P(+z)=\frac12.$$

These are theoretical checks, not substitutes for simulating the density matrices.

**Prediction A (purity):** How should purity vary as phase uncertainty increases? Predict both limiting values.

**Prediction A — Part 2**

> Write your response here.

**Prediction B (measurements):** Which of $P(+x)$, $P(+y)$, and $P(+z)$ should change with phase spread? Why?

**Prediction B — Part 2**

> Write your response here.

### Code Stem — Measurement projectors and purity

In [ ]:
plus_x = (up + down).unit()
plus_y = (up + 1j * down).unit()
P_x = qt.ket2dm(plus_x)
P_y = qt.ket2dm(plus_y)
P_z = qt.ket2dm(up)


def analyze_density_matrix(rho):
    purity = (rho * rho).tr().real
    p_x = (rho * P_x).tr().real
    p_y = (rho * P_y).tr().real
    p_z = (rho * P_z).tr().real
    return purity, p_x, p_y, p_z

print("Example:", analyze_density_matrix(rho))

# TODO: reuse the delta sweep from Part 1 and store all four results.

### Figure 3 — Two-panel result

**Top:** numerical purity and theoretical $\frac12(1+\gamma^2)$. **Bottom:** numerical $P(+x),P(+y),P(+z)$ and theoretical predictions as useful. Label axes (radians/degrees), legends and curves.

In [ ]:
# TODO: generate Figure 3 with two stacked panels.
# Keep analytical predictions separate from simulated values.

**Figure 3 caption**

> [Explain the changing purity and the differing basis sensitivities.]

### Verification — Part 2

1. Compare the limits $\Delta=0$ and $\pi$ with pure $|+x\rangle$ and the maximally mixed state.
2. At $\Delta=\pi/2$, use $\gamma=2/\pi$ to calculate the expected purity and $P(+x)$. Report both numerical errors.
3. Check probabilities are in $[0,1]$, and confirm $P(+x)+P(-x)=1$ for a selected state.
4. Verify $\mathcal P=\frac12(1+|\mathbf r|^2)$ numerically for one density matrix.
5. Record a table with purity and the three measurement probabilities at $\Delta=0,\pi/2,\pi$.

In [ ]:
# TODO: print the three-row result table and verification comparisons.
# TODO: calculate a P(-x) projector and check normalization.

**Verification and predictions revisited — Part 2**

> [Give your calculations and compare predictions with observations.]

### Optional: Rotating the mean phase

If desired, center phases at $\phi_0=\pi/2$ instead of 0. Investigate whether the roles of $x$ and $y$ probabilities exchange while purity stays the same.

In [ ]:
# OPTIONAL TODO: offset the ensemble phases and compare results.

### Physics Analysis — Part 2 (write independently)

**Question 3.** How does purity change as the average Bloch vector shrinks? Why does a completely phase-randomized **equatorial** ensemble approach purity $1/2$ rather than zero?

**Answer to Question 3**

> Write your response here.

**Question 4.** Why does $P(+x)$ depend on phase spread while $P(+z)$ does not? Explain using the diagonal and off-diagonal elements of $\rho$, not only the graph.

**Answer to Question 4**

> Write your response here.

---
## Part 3 — Quantum Interference
### Activity 3: Phase uncertainty in a Stern–Gerlach interferometer

**Physics model:** An initial $|+z\rangle$ beam is coherently separated into $|+x\rangle$ and $|-x\rangle$ paths, given a relative phase, ideally recombined into one spatial mode, and finally analyzed along $z$.

$$|\psi(\delta)\rangle=\frac1{\sqrt2}(|+x\rangle+e^{i\delta}|-x\rangle),\quad P(+z)=\frac12(1+\cos\delta).$$

![Ideal coherent Stern–Gerlach interferometer](attachment:coherent_stern_gerlach_recombination.png)

*Apparatus A: Ideal coherent recombination. At zero phase difference the final $+z$ output has probability 1.*

When the actual phase is $\delta+\epsilon$, with $\epsilon$ uniform over $[-\Delta,\Delta]$, the ideal ensemble prediction is

$$P(+z)=\frac12\left[1+\frac{\sin\Delta}{\Delta}\cos\delta\right],\qquad V=\frac{P_{\max}-P_{\min}}{P_{\max}+P_{\min}}=\left|\frac{\sin\Delta}{\Delta}\right|.$$

The visibility is also $V=2|\rho_{12}|$ in the **intermediate path basis**, not automatically in every basis.

![Phase-noisy Stern–Gerlach interferometer](attachment:partial_coherence_stern_gerlach_interferometer.png)

*Apparatus B: Uncontrolled phase fluctuations reduce the interference-fringe contrast.*

### Initial Predictions — Part 3

**Prediction A:** For $\Delta=0$, predict $P(+z)$ at $\delta=0,\pi/2,\pi$ and sketch/describe the expected fringe for $0\leq\delta\leq2\pi$.

**Prediction A — Part 3**

> Write your response here.

**Prediction B:** As $\Delta$ increases through $0,\pi/2,\pi$, what happens to fringe maxima, minima, and visibility? What remains at $P(+z)=1/2$?

**Prediction B — Part 3**

> Write your response here.

### Code Stem — Interferometer

In [ ]:
plus_x = (up + down).unit()
minus_x = (up - down).unit()
P_out = qt.ket2dm(up)  # final +z detector


def path_state(phase):
    return (plus_x + np.exp(1j * phase) * minus_x).unit()


def output_probability(delta, Delta, N=1000, seed=42):
    rng = np.random.default_rng(seed)
    random_shifts = rng.uniform(-Delta, Delta, N)
    rho_out = sum(
        (qt.ket2dm(path_state(delta + eps)) for eps in random_shifts),
        start=0
    ) / N
    return (rho_out * P_out).tr().real

print("Fully coherent:", output_probability(0, 0))

# TODO: sweep controlled delta over [0, 2*pi] for Delta=0, pi/2, pi.
# Optimize if needed, but compute numerical ensemble results rather
# than substituting an analytical curve.

### Figure 4 — Interference fringes

Plot three simulated curves versus controlled $\delta$, overlay analytical predictions, calculate the visibility for each phase spread, and label the resulting figure.

In [ ]:
# TODO: generate Figure 4 (numeric + analytic fringes).
# TODO: estimate P_max, P_min, and visibility for each curve.

**Figure 4 caption**

> [Explain how relative phase and phase uncertainty influence fringe contrast.]

### Verification — Part 3

1. At $\Delta=0$, compare numerical $P(+z)$ at $\delta=0,\pi/2,\pi$ with theory.
2. At $\Delta=\pi$, check that $P(+z)\approx1/2$ for all controlled phases, up to sampling fluctuations.
3. At $\Delta=\pi/2$, compare simulated visibility with $V=2/\pi$; report absolute error.
4. Compare your visibility with $2|\rho_{12}|$ from the mathematically equivalent phase ensemble in Part 1. **State the basis used** in each computation. The Part 3 path basis is $x$; Part 1 used the $z$ basis.

In [ ]:
# TODO: numerical verification table of fringes and visibility errors.

**Verification and predictions revisited — Part 3**

> [Provide numerical comparisons and revisit your two initial predictions.]

### Connecting to welcher Weg — Conceptual extension (no new code required)

Phase averaging and a path detector can produce the same reduced spin state through **different mechanisms**. An ideal path detector becomes correlated with the arm taken; the joint spin–detector state can remain pure while the spin subsystem becomes mixed.

![Which-path detectors in a Stern–Gerlach interferometer](attachment:which_path_information_diagram.png)

*Apparatus C: Orthogonal detector records make the paths distinguishable and eliminate reduced-spin interference.*

$$|\Psi\rangle=\frac1{\sqrt2}(|+x\rangle|D_+\rangle+|-x\rangle|D_-\rangle),$$

$$\rho_{\mathrm{spin}}=\frac12\begin{pmatrix}1&\langle D_-|D_+\rangle\\\langle D_+|D_-\rangle&1\end{pmatrix}.$$

If the detector states are orthogonal, their overlap is zero and the reduced-spin off-diagonal entries vanish. The detector need not be read by a person. **Assumption:** the interferometer must coherently overlap spatial modes; merely bringing paths near each other does not guarantee interference.

### Physics Analysis — Part 3 (write independently)

**Question 5.** Explain why increasing random-phase spread reduces fringe visibility even though each path is populated with probability $1/2$. Relate this to the off-diagonal density-matrix terms in the $x$ path basis.

**Answer to Question 5**

> Write your response here.

**Question 6.** How can phase averaging and entanglement with a which-path detector produce the same reduced spin density matrix? If so, could a measurement on the **spin alone** distinguish them?

**Answer to Question 6**

> Write your response here.

---
## AI Development Record — Complete at the end of your notebook

Provide **one meaningful, genuine AI interaction per activity**. If you did not use AI in an activity, describe your independent code-development and checking process. **Do not paste full chat logs or unrelated/private material.**

For each record: objective; actual prompt; AI contribution; how you verified, corrected, or revised the output. AI may assist with coding but **must not write your physics analysis answers**.

### Activity 1 — Constructing Mixed States

**Objective:** [Write here]

**Actual prompt:**

> [Paste one meaningful prompt here]

**AI contribution:** [Write here]

**Verification / revision:** [Write here]

*No AI used? Replace this entry with your independent coding and verification process.*

### Activity 2 — Measuring Coherence

**Objective:** [Write here]

**Actual prompt:**

> [Paste one meaningful prompt here]

**AI contribution:** [Write here]

**Verification / revision:** [Write here]

*No AI used? Replace this entry with your independent coding and verification process.*

### Activity 3 — Quantum Interference

**Objective:** [Write here]

**Actual prompt:**

> [Paste one meaningful prompt here]

**AI contribution:** [Write here]

**Verification / revision:** [Write here]

*No AI used? Replace this entry with your independent coding and verification process.*

### Final AI Reflection (approximately 100 words)

Reflect on (1) how AI helped or accelerated your investigation, (2) one error, limitation, or uncertainty you personally investigated, and (3) one physics insight gained by interpreting results rather than simply running code.

**Final reflection**

> [Write approximately 100 words here in your own words.]

---
## Final Submission Checklist

- [ ] Analytical warm-up completed and checked using QuTiP.
- [ ] Initial predictions **and** revisited predictions for each of Parts 1–3.
- [ ] **Figure 1:** individual Bloch states and average vectors for three phase spreads.
- [ ] **Figure 2:** off-diagonal coherence versus phase spread, numerical + analytical.
- [ ] **Figure 3:** two-panel purity and spin-measurement probabilities.
- [ ] **Figure 4:** interference fringes and analytical predictions.
- [ ] Required matrices, tables, and quantitative verification steps included.
- [ ] Six numbered physics analysis answers completed independently.
- [ ] Three AI Development Records (or independent-development equivalents).
- [ ] Final AI Reflection completed.
- [ ] Figures have labels, units where appropriate, legends, and explanatory captions.
- [ ] Notebook runs top-to-bottom in a clean kernel after installing dependencies.

**Evaluation:** physics predictions and interpretation 40%; visualizations 30%; calculations and verification 20%; clarity, reproducibility, and AI Development Record 10%.